In [ ]:
#1. Imports
# general imports
from pathlib import Path
import numpy as np
import pandas as pd
import re
from tqdm import tqdm
import statsmodels.api as sm

np.random.seed(42)

from nispace.datasets import fetch_reference
from nispace.plotting import view_surf
from nispace.workflows import group_comparison

In [ ]:
#2. Settings
DATA_PATH = Path("../../data/df1.csv")

PARCELLATION = "DesikanKilliany"

DK_REGIONS = [
    "bankssts","caudalanteriorcingulate","caudalmiddlefrontal","cuneus","entorhinal",
    "fusiform","inferiorparietal","inferiortemporal","isthmuscingulate","lateraloccipital",
    "lateralorbitofrontal","lingual","medialorbitofrontal","middletemporal","parahippocampal",
    "paracentral","parsopercularis","parsorbitalis","parstriangularis","pericalcarine",
    "postcentral","posteriorcingulate","precentral","precuneus","rostralanteriorcingulate",
    "rostralmiddlefrontal","superiorfrontal","superiorparietal","superiortemporal",
    "supramarginal","frontalpole","temporalpole","transversetemporal","insula",
]

DEMO_COLS = ["PATNO", "CONCOHORT", "age", "SEX", "agediag", "subgroup", "PRIMDIAG"]

CONTRASTS = {
    "De Novo PD vs HC": (1.0, 2.0),
    "Prodromal PD vs HC": (4.0, 2.0),
    "De Novo PD vs Prodromal PD": (1.0, 4.0)
}

GROUP_LABELS = {
    1.0: "De Novo PD",
    2.0: "HC",
    4.0: "Prodromal PD"
}

In [ ]:
#Choosing the reference maps
#Going by partial matching - MIGHT CHANGE
SELECTED_REFERENCE_MAPS = [
    "target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019",
    "target-NMDA_tracer-ge179_n-29_dx-hc_pub-galovic2021",
    "target-GABAa_tracer-flumazenil_n-6_dx-hc_pub-dukart2018",
    "target-FDOPA_tracer-fluorodopa_n-12_dx-hc_pub-garciagomez2018",
    "target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017",
    "target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015",
    "target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018",
    "target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012",
    "target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012",
    "target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012",
    "target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017",
    "target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012",
    "target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017",
    "target-VAChT_tracer-feobv_n-18_dx-hc_pub-aghourian2017",
]

SELECTED_REFERENCE_CONTAINS = []

N_PERM = 10000

In [ ]:
#3. Helper functions
def get_dk_volume_columns(df: pd.DataFrame) -> list[str]:
    """Return DK cortical volume columns only."""
    dk_region_pattern = "|".join(map(re.escape, DK_REGIONS))
    regex = re.compile(rf"(?i)^(lh|rh)_({dk_region_pattern})_volume$")
    return [c for c in df.columns if regex.match(c)]


def pick_etiv_column(df: pd.DataFrame) -> str:
    """Find eTIV column in the subject dataframe."""
    candidates = [
        "eTIV",
        "EstimatedTotalIntraCranialVol",
        "IntraCranialVol",
        "intracranialvolume",
        "EstimatedTotalIntracranialVol",
    ]
    exact = [c for c in candidates if c in df.columns]
    if exact:
        return exact[0]
    for c in df.columns:
        cn = str(c).strip().lower().replace("-", "").replace("_", "")
        if cn in {"etiv", "estimatedtotalintracranialvol", "intracranialvol", "intracranialvolume"}:
            return c
    raise ValueError("Could not find an eTIV column in the dataframe.")


def rename_to_nispace(col: str) -> str:
    """Convert lh/rh volume column names to NiSpace DK parcel names."""
    match = re.match(r"(lh|rh)_(.+)_volume", col, re.IGNORECASE)
    if not match:
        return col
    hemi, region = match.groups()
    hemi_letter = "L" if hemi.lower() == "lh" else "R"
    return f"hemi-{hemi_letter}_lab-{region.lower()}"


def prepare_brain_and_design(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Prepare Y matrix and design matrix for cortical volume analysis."""
    etiv_col = pick_etiv_column(df)
    dk_cols = get_dk_volume_columns(df)

    print("Number of DK volume columns:", len(dk_cols))
    print("Using eTIV column:", etiv_col)

    demo_cols = [c for c in DEMO_COLS if c in df.columns]
    fs_col = "Field Strength"
    fs_cols = [fs_col] if fs_col in df.columns else []
    df_dk = df[demo_cols + fs_cols + [etiv_col] + dk_cols].copy()

    # brain matrix
    Y = df_dk[dk_cols].apply(pd.to_numeric, errors="coerce")
    Y.index = df_dk["PATNO"].astype(str)

    # design matrix
    design = pd.DataFrame(index=Y.index)
    design["CONCOHORT"] = pd.to_numeric(df_dk["CONCOHORT"], errors="coerce").values
    design["age"] = pd.to_numeric(df_dk["age"], errors="coerce").values
    design["SEX"] = df_dk["SEX"].values
    design["eTIV"] = pd.to_numeric(df_dk[etiv_col], errors="coerce").values
    if fs_col in df_dk.columns:
        fs_vals = pd.to_numeric(df_dk[fs_col], errors="coerce")
        if fs_vals.isna().all():  # string column ("1.5T" / "3T") — encode as codes
            fs_vals = pd.Categorical(df_dk[fs_col]).codes.astype(float)
            fs_vals[fs_vals < 0] = np.nan
        design["field_strength"] = fs_vals.values
    else:
        print("WARNING: 'Field Strength' column not found — covariate omitted.")

    return Y, design


def align_y_to_reference(Y: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    """Rename and reorder Y columns to match reference parcel columns."""
    Y_renamed = Y.copy()
    Y_renamed.columns = [rename_to_nispace(c) for c in Y.columns]

    Y_aligned = Y_renamed.reindex(columns=ref_df.columns).copy()

    print("Y aligned shape:", Y_aligned.shape)
    print("Any missing values after parcel reindex?", Y_aligned.isna().any().any())

    missing_cols = Y_aligned.columns[Y_aligned.isna().all(axis=0)].tolist()
    if missing_cols:
        print("Warning: these parcels are entirely missing in Y:")
        for col in missing_cols:
            print(" -", col)

    return Y_aligned


def run_parcelwise_ttest(
    Y_aligned: pd.DataFrame,
    design: pd.DataFrame,
    g1: float,
    g2: float,
) -> pd.DataFrame:
    """
    Parcelwise group comparison adjusted for eTIV, age, SEX using OLS:
        parcel ~ group + eTIV + age + SEX

    With coding {g1: 0, g2: 1}, a positive t-value means higher adjusted volume in g2 than g1.
    """
    mask = design["CONCOHORT"].astype(float).isin([g1, g2])
    y_sub = Y_aligned.loc[mask].copy()
    d_sub = design.loc[mask].copy()

    d_sub = d_sub.copy()
    d_sub["group01"] = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1})

    if d_sub["SEX"].dtype == object:
        d_sub["SEX"] = pd.Categorical(d_sub["SEX"]).codes
    d_sub["SEX"] = pd.to_numeric(d_sub["SEX"], errors="coerce")
    d_sub["age"] = pd.to_numeric(d_sub["age"], errors="coerce")
    d_sub["eTIV"] = pd.to_numeric(d_sub["eTIV"], errors="coerce")
    d_sub["group01"] = pd.to_numeric(d_sub["group01"], errors="coerce")
    has_fs = "field_strength" in d_sub.columns
    if has_fs:
        d_sub["field_strength"] = pd.to_numeric(d_sub["field_strength"], errors="coerce")

    def parcel_hemi(parcel_name: str) -> str:
        p = str(parcel_name)
        if p.startswith("hemi-L"):
            return "L"
        if p.startswith("hemi-R"):
            return "R"
        return "NA"

    t_vals, p_vals, dfs = [], [], []

    for parcel in y_sub.columns:
        tmp_dict = {
            "y": pd.to_numeric(y_sub[parcel], errors="coerce"),
            "group01": d_sub["group01"],
            "eTIV": d_sub["eTIV"],
            "age": d_sub["age"],
            "SEX": d_sub["SEX"],
        }
        if has_fs:
            tmp_dict["field_strength"] = d_sub["field_strength"]
        tmp = pd.DataFrame(tmp_dict, index=y_sub.index).dropna()

        if tmp.shape[0] < 5 or tmp["group01"].nunique() < 2:
            t_vals.append(np.nan)
            p_vals.append(np.nan)
            dfs.append(np.nan)
            continue

        cov_cols = ["group01", "eTIV", "age", "SEX"] + (["field_strength"] if has_fs else [])
        X = sm.add_constant(tmp[cov_cols])
        model = sm.OLS(tmp["y"], X).fit()

        t_vals.append(model.tvalues.get("group01", np.nan))
        p_vals.append(model.pvalues.get("group01", np.nan))
        dfs.append(model.df_resid)

    df_out = pd.DataFrame(
        {
            "Tvalue": t_vals,
            "pvalue": p_vals,
            "df": dfs,
            "hemi": [parcel_hemi(c) for c in Y_aligned.columns],
        },
        index=Y_aligned.columns,
    )
    return df_out


def build_nispace_design(d_sub: pd.DataFrame, g1: float, g2: float) -> pd.DataFrame:
    """
    Build design DataFrame for NiSpace:
    groups: 0 = g1, 1 = g2
    covariates: age, SEX, eTIV
    """
    groups01 = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1}).astype(int)

    design_dict = {
        "groups": groups01,
        "age": pd.to_numeric(d_sub["age"], errors="coerce"),
        "SEX": d_sub["SEX"],
        "eTIV": pd.to_numeric(d_sub["eTIV"], errors="coerce"),
    }
    if "field_strength" in d_sub.columns:
        design_dict["field_strength"] = pd.to_numeric(d_sub["field_strength"], errors="coerce")

    design_df = pd.DataFrame(design_dict, index=d_sub.index)

    if design_df["SEX"].dtype == object:
        design_df["SEX"] = pd.Categorical(design_df["SEX"]).codes

    design_df["SEX"] = pd.to_numeric(design_df["SEX"], errors="coerce")

    return design_df


def run_group_comparisons(
    Y_aligned: pd.DataFrame,
    design: pd.DataFrame,
    ref_df: pd.DataFrame,
    contrasts: dict[str, tuple[float, float]],
    labels: dict[float, str],
    n_perm: int = 10000,
) -> tuple[pd.DataFrame, dict]:
    """Run NiSpace group comparisons for all requested contrasts."""
    all_rows = []
    outputs = {}

    for contrast_name, (g1, g2) in contrasts.items():
        mask = design["CONCOHORT"].astype(float).isin([g1, g2])

        y_sub = Y_aligned.loc[mask].copy()
        d_sub = design.loc[mask].copy()

        design_df = build_nispace_design(d_sub, g1, g2)

        covariate_cols = [c for c in ["groups", "age", "SEX", "eTIV", "field_strength"]
                          if c in design_df.columns]
        keep = ~(
            y_sub.isna().any(axis=1)
            | design_df[covariate_cols].isna().any(axis=1)
        )

        y_sub = y_sub.loc[keep]
        design_df = design_df.loc[keep]
        d_sub = d_sub.loc[keep]

        print("\n--------------------------")
        print(f"Contrast: {contrast_name}")
        print("Counts:", d_sub["CONCOHORT"].astype(float).map(labels).value_counts().to_dict())
        print("Y shape:", y_sub.shape)
        print("Design shape:", design_df.shape)
        print("Reference maps:", len(ref_df.index))

        np.random.seed(42)

        colocs, pvals, qvals, nsp = group_comparison(
            y=y_sub,
            x=ref_df,
            parcellation=PARCELLATION,
            design=design_df,
            comparison_method="hedges(a,b)",
            colocalization_method="spearman",
            n_perm=n_perm,
            n_proc=-1,
            verbose=True,
        )

        outputs[contrast_name] = {
            "colocs": colocs,
            "p": pvals,
            "q": qvals,
            "nsp": nsp,
        }

        rho = np.asarray(colocs).ravel()
        p = np.asarray(pvals).ravel()
        q = np.asarray(qvals).ravel()

        df_out = pd.DataFrame(
            {
                "reference_map": ref_df.index,
                "rho": rho,
                "p": p,
                "q": q,
                "contrast": contrast_name,
            }
        )

        all_rows.append(df_out)

    df_all = pd.concat(all_rows, ignore_index=True).sort_values(["contrast", "q", "p"])
    return df_all, outputs

In [ ]:
#4. Load Data
df = pd.read_csv(DATA_PATH, low_memory=False)

df.head(5)

In [ ]:
etiv_col = pick_etiv_column(df)
print(f"eTIV column: {etiv_col}")
print(df[etiv_col].describe())

In [ ]:
Y, design = prepare_brain_and_design(df)

print("\nY shape:", Y.shape)
print("Design shape:", design.shape)
print("\nCONCOHORT counts:")
print(design["CONCOHORT"].value_counts(dropna=False))

In [ ]:
#5. Fetch and select reference maps
df_reference_desikan = fetch_reference(
    "pet",
    collection="UniqueTracers",
    parcellation=PARCELLATION,
    print_references=True,
)

print("\nFull reference shape:", df_reference_desikan.shape)
print("Available reference maps:")
print(df_reference_desikan.index.tolist())

In [ ]:
df_reference_selected = df_reference_desikan[
    df_reference_desikan.index.get_level_values("map").isin(SELECTED_REFERENCE_MAPS)
]

if SELECTED_REFERENCE_CONTAINS:
    mask = df_reference_selected.index.get_level_values("map").str.contains(
        "|".join(SELECTED_REFERENCE_CONTAINS),
        case=False,
        na=False
    )
    df_reference_selected = df_reference_selected[mask]

reference_table = df_reference_selected.index.to_frame(index=False)
reference_table.columns = ["Set", "Map"]
reference_table = reference_table.reset_index(drop=True)
reference_table.index = reference_table.index + 1

print("\nSelected reference shape:", df_reference_selected.shape)
print("Selected reference maps:")
print(reference_table.to_string())

In [ ]:
#6. Align Y to reference parcels
Y_aligned = align_y_to_reference(Y, df_reference_selected)

common_idx = Y_aligned.index.intersection(design.index)
Y_aligned = Y_aligned.loc[common_idx].copy()
design = design.loc[common_idx].copy()

print("\nFinal aligned Y shape:", Y_aligned.shape)
print("Final aligned design shape:", design.shape)

In [ ]:
# Save subject-level aligned DK volume table for R/ggseg visualization
df_ggseg = (
    design.loc[common_idx, [c for c in ["CONCOHORT", "age", "SEX", "eTIV", "field_strength"]
                             if c in design.columns]]
    .copy()
    .assign(PATNO=common_idx)
)

_demo_cols = [c for c in ["PATNO", "CONCOHORT", "age", "SEX", "eTIV", "field_strength"]
              if c in df_ggseg.columns]
df_ggseg = pd.concat(
    [
        df_ggseg[_demo_cols],
        Y_aligned
    ],
    axis=1
)

df_ggseg.to_csv("../../data/df1_id_volume_cortical_aligned.csv", index=False)
print("Saved:", "../../data/df1_id_volume_cortical_aligned.csv")
print(df_ggseg.shape)
display(df_ggseg.head())

In [ ]:
#7. (optional) parcelwise t-test example
parcelwise_ttests = {}

for contrast_name, (g1, g2) in CONTRASTS.items():
    df_ttest = run_parcelwise_ttest(Y_aligned, design, g1, g2)
    df_ttest_reset = df_ttest.reset_index().rename(columns={"index": "parcel"})

    outname = f"../../results/{contrast_name.lower()}_parcelwise_ttest_volume_cortical.csv"
    df_ttest_reset.to_csv(outname, index=False)

    parcelwise_ttests[contrast_name] = df_ttest_reset

    print(f"Saved: {outname}")
    display(df_ttest_reset.head())

    print(f"\nTop 5 parcels by p-value ({contrast_name}):")
    print(df_ttest.sort_values("pvalue").head(5))

    view_surf(
        df_ttest.loc[df_ttest["hemi"] == "L", "Tvalue"],
        parcellation=PARCELLATION,
        template="fsaverage",
        hemi="L",
    )

    view_surf(
        df_ttest.loc[df_ttest["hemi"] == "R", "Tvalue"],
        parcellation=PARCELLATION,
        template="fsaverage",
        hemi="R",
    )

In [ ]:
#8. Running NiSpace group comparisons
df_all, outputs = run_group_comparisons(
    Y_aligned=Y_aligned,
    design=design,
    ref_df=df_reference_selected,
    contrasts=CONTRASTS,
    labels=GROUP_LABELS,
    n_perm=N_PERM
)

In [ ]:
#9. Checking the results

print("\n=============================")
print("Combined results")
print("\n=============================")
display(df_all.head(10))

In [ ]:
df_all.to_csv("../../results/nispace_group_comparison_results_volumes_cortical.csv", index=False)

In [ ]:
for contrast_name, res in outputs.items():
    print("\n=============================")
    print(f"Contrast: {contrast_name}")
    print("\n=============================")

    colocs = res["colocs"]
    pvals = res["p"]
    qvals = res["q"]

    print("Top 5 lowest q-values")
    print(qvals.T["mean"].sort_values().head(5))

    display("Colocalization:", colocs)
    display("p values:", pvals)
    display("q values:", qvals)

print("\nTotal NaNs in selected reference:", df_reference_selected.isna().sum().sum())
print("NaNs per selected map:")
display(df_reference_selected.isna().sum(axis=1).sort_values(ascending=False))

for contrast_name in df_all["contrast"].unique():
    print(f"\nTop 5 results for {contrast_name}:")
    display(
        df_all[df_all["contrast"] == contrast_name]
        .sort_values("q").head(5)
    )

## Advanced analyses: Group Comparison using single-subject z-scores
- **To interpret it correctly:**
1. **What zscore(a, b) means here**
    - It uses group b as the reference (controls) to compute z-scores
    - with our encoding "groups: 0=g1, 1=g2", that means: a= group 0(g1), b= group 1(g2)
    - so for PD_vs_HC with (g1, g2) = (PD, HC) you get PD z-scored relative to HC - which is the "standard" JuSPace style
2. **Make sure the "control" is always the second group**
    - Your contrasts already do that for *_vs_HC (HC is g2)
    - for PD_vs_Prodromal, neither is a true "healthy control"; the z-score will still compute, but interpret it as **PD standardized relative to Prodromal**, not "abnormality vs healthy"

In [ ]:
zscore_outputs = {}

for contrast_name, (gA, gB) in CONTRASTS.items():
    mask = design["CONCOHORT"].astype(float).isin([gA, gB])
    y = Y_aligned.loc[mask].copy()
    d = design.loc[mask].copy()

    _cov_cols = [c for c in ["age", "SEX", "eTIV", "field_strength"] if c in d.columns]
    keep = ~(y.isna().any(axis=1) | d[_cov_cols].isna().any(axis=1))
    y = y.loc[keep]
    d = d.loc[keep]

    design_sub = build_nispace_design(d, gA, gB)

    print(f"\n{contrast_name}")
    print("Counts:", d["CONCOHORT"].astype(float).map(GROUP_LABELS).value_counts().to_dict())

    colocs, pvals, qvals, nsp = group_comparison(
        y=y,
        x=df_reference_selected,
        parcellation=PARCELLATION,
        design=design_sub,
        comparison_method="zscore(a,b)",
        colocalization_method="spearman",
        n_perm=N_PERM,
        n_proc=-1,
        verbose=False,
        plot_design=False,
    )

    zscore_outputs[contrast_name] = {
        "colocs": colocs,
        "p": pvals,
        "q": qvals,
        "nsp": nsp,
    }

- The NiSpace z-score group comparison produces:
    - One colocalization value per reference map per contrast
    - plus null/permutation significance
    - DOES NOT PRODUCE --> one rho per subject per map

"This is the same NiSpace z-score colocalization analysis as the Destrieux example, adapted to the DK atlas and run separately for each pairwise group contrast with covariate adjustment"

In [ ]:
for contrast, res in zscore_outputs.items():
    print(f"\n{contrast}")
    display("Colocalization:", res["colocs"].head(3))
    display("P values:", res["p"].head())
    display("Q values:", res["q"].head())

In [ ]:
import pandas as pd

all_rows = []

for contrast, res in zscore_outputs.items():
    colocs = res["colocs"].copy()

    # make sure subject index has a name
    if colocs.index.name is None:
        colocs.index.name = "PATNO"

    # stack all column levels into rows
    colocs_long = colocs.stack(list(range(colocs.columns.nlevels))).reset_index()

    # rename the stacked value column
    colocs_long = colocs_long.rename(columns={0: "colocalization"})

    # build a single map label from all non-PATNO, non-colocalization columns
    meta_cols = [c for c in colocs_long.columns if c not in ["PATNO", "colocalization"]]

    colocs_long["map"] = colocs_long[meta_cols].astype(str).agg(" | ".join, axis=1)

    # keep only the relevant columns
    colocs_long = colocs_long[["PATNO", "map", "colocalization"]].copy()
    colocs_long["contrast"] = contrast

    all_rows.append(colocs_long)

nispace_df = pd.concat(all_rows, ignore_index=True)

In [ ]:
print(nispace_df.head())
print(nispace_df.shape)
print(nispace_df["contrast"].unique())

In [ ]:
clinical_df = df[["PATNO", "updrs3_score", "moca", "gds", "SEX", "age"]].copy()

In [ ]:
nispace_df["PATNO"] = nispace_df["PATNO"].astype(str)
clinical_df["PATNO"] = clinical_df["PATNO"].astype(str)

In [ ]:
merged_df = nispace_df.merge(clinical_df, on="PATNO", how="inner")

In [ ]:
print(merged_df.head())
print(merged_df.shape)

In [ ]:
# 10. Save merged dataset
merged_df.to_csv("../../data/merged_df_volume_cortical.csv", index=False)

print("\nSaved as merged_df_volume_cortical.csv")